[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/mlops-internship-in-a-book/blob/main/notebooks/week5_retraining_pipeline_STARTER.ipynb)


[![Get the Book on Selar](https://img.shields.io/badge/Get%20the%20full%20book-Selar-1E7A34?style=for-the-badge)](https://selar.com/7m27877571)

# Week 5 -- The Retraining Pipeline (STARTER)
### The MLOps Internship | DataFlow Infrastructure

**Dataset:** CreditDefault-NG (UCI Credit Card Default)

**This week:** Airflow retraining DAG, trigger strategies, model validation gate, automated rollback

STARTER -- complete each exercise before moving on.


In [ ]:
# -- Colab Setup (skip if running locally) -------------------------
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/mlops-internship-in-a-book.git mlops-book
    %cd mlops-book
    !pip install -r requirements.txt -q
os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('datasets', exist_ok=True)
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds ------------------------------------------
# SEED=42 ensures every random operation gives the same result on every run.
# Set seeds BEFORE any data loading, model creation, or training.
import random, numpy as np
SEED = 42
random.seed(SEED)       # Python random
np.random.seed(SEED)    # NumPy random (used by sklearn)
print(f'Seeds fixed: {SEED}')


## Learning Objectives

- Design the retraining trigger strategy: schedule, performance, and drift triggers
- Build an Airflow DAG for weekly retraining with check, retrain, validate, promote/rollback tasks
- Implement the model validation gate: new model must beat current on all LIMIT_BAL slices
- Add automated rollback: validation failure archives the model and pages on-call
- Test the full DAG by simulating the Week 2 drift event



---

## MONDAY -- "Retraining Trigger Strategies"


Three strategies. Schedule-based: retrain every N days regardless of performance -- simple but wastes compute on stable periods. Performance-based: retrain when AUC drops below the SLA floor -- catches model degradation directly. Drift-based: retrain when PSI exceeds the SLA threshold -- catches data drift before performance drops, no labels needed. The credit model uses all three: the first to fire triggers retraining.

Pause and Predict -- which trigger would have caught the Q4 drift earliest? The schedule trigger (if set to monthly), the performance trigger (requires labelled Q4 data), or the drift trigger (based on PSI, which you computed in Week 2)?


### Exercise 5.1 -- Trigger matrix

Build a 3x4 matrix: trigger type x scenario (normal/AUC degradation/drift/both). For each cell: does retraining trigger? Which check fires first?


In [ ]:
# Exercise 5.1: Trigger matrix
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Monday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Monday: Retraining Trigger Strategies --
def should_retrain(current_auc, auc_threshold,
                   max_psi, psi_threshold,
                   days_since_retrain, max_days):
    """Check all three retraining triggers. Returns (retrain: bool, reason: str)"""
    if current_auc < auc_threshold:                        # trigger 1: quality floor breached
        return True, f'AUC {current_auc:.4f} below SLA floor {auc_threshold}'
    if max_psi > psi_threshold:                            # trigger 2: drift threshold breached
        return True, f'PSI {max_psi:.4f} exceeds SLA threshold {psi_threshold}'
    if days_since_retrain >= max_days:                     # trigger 3: scheduled cadence
        return True, f'Scheduled: {days_since_retrain} days since last retrain'
    return False, 'All checks pass -- no retraining needed'

# Simulate current state after Week 2 findings:
print(should_retrain(
    current_auc=0.74,       # holdout AUC -- below the 0.80 floor
    auc_threshold=0.80,
    max_psi=0.38,           # LIMIT_BAL PSI from Week 2
    psi_threshold=0.25,
    days_since_retrain=180, # six months without retraining
    max_days=90,
))


### Expected Output (exact — verified by running this function directly)

```
(True, 'AUC 0.7400 below SLA floor 0.8')
```
Notice the function checks triggers in order and returns on the *first* one met — with these
inputs, both the AUC floor and the PSI threshold are breached, but the function reports only
the AUC reason. This is worth discussing: should a retraining trigger report *all* breached
conditions, not just the first? Consider that a design choice, not a bug, and defend or change
it in your own implementation.


### Monday Morning Moment

*Slack -- Monday, 3:00pm.*

**Sola Fashola:** Schedule trigger only, or drift trigger too?

**Temi Adeyemi:** Starting with schedule and performance. Adding drift trigger in Week 9 when the Evidently monitoring is confirmed accurate.

**Sola Fashola:** Correct order. A drift trigger that fires on monitoring noise will retrain every day.

**Temi Adeyemi:** Minimum viable DAG?

**Sola Fashola:** Four tasks. Trigger check, retrain, validate, promote or rollback. Every task logs to MLflow. Every failure alerts. No silent failures.

**Dr. Emeka Obi:** Readable DAG. Every task function needs a docstring. I review it on Friday.




---

## TUESDAY -- "Building the Airflow DAG"


An Airflow DAG defines a pipeline as tasks with dependencies. Each task is a Python function decorated with @task. The DAG runs on a cron schedule. For the credit model: every Monday at 2am, check triggers, retrain if needed, validate the new model, and promote to Staging or rollback.

Pause and Predict -- draw the dependency graph before writing code. Which tasks can run in parallel? What happens if train_model raises an exception -- does validate_model still run? What should happen?


### Exercise 5.2 -- DAG implementation

Implement all four task functions. Each must: log to MLflow, handle the no-retrain short-circuit, raise on failure so Airflow marks the task failed.


In [ ]:
# Exercise 5.2: DAG implementation
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Tuesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Tuesday: Building the Airflow DAG --
from airflow.decorators import dag, task
import pendulum

@dag(dag_id='credit_model_retraining',
     schedule_interval='0 2 * * MON',  # Mondays at 2am
     start_date=pendulum.datetime(2026, 6, 8),
     catchup=False, tags=['credit', 'retraining'])
def credit_retraining_pipeline():
    """Weekly automated retraining pipeline for the credit default model."""

    @task()
    def check_triggers() -> dict:
        """Check all retraining triggers. Returns trigger status dict."""
        pass  # Exercise 5.2: implement this

    @task()
    def fetch_training_data(trigger: dict) -> str:
        """Fetch and validate latest training data. Returns data path."""
        pass

    @task()
    def train_model(data_path: str) -> str:
        """Train model on latest data. Returns MLflow run_id."""
        pass

    @task()
    def validate_and_promote(run_id: str) -> dict:
        """Validate new model vs current production. Promote or rollback."""
        pass

    t = check_triggers()
    d = fetch_training_data(t)
    m = train_model(d)
    validate_and_promote(m)

credit_pipeline = credit_retraining_pipeline()


### Expected Output (illustrative — Airflow UI)

Running `airflow dags list` after this DAG is registered shows:

```
dag_id                      | schedule       | owner
credit_model_retraining     | 0 2 * * MON    | dataflow-ml
```
The four `@task()` functions currently `pass` — Exercise 5.2 asks you to implement each one.
Until then, `airflow dags trigger credit_model_retraining` will register the run but every
task will fail immediately (a `pass` inside a typed-return function is a good way to notice
quickly which tasks are still unimplemented).



---

## WEDNESDAY -- "The Model Validation Gate"


The validation gate prevents a worse model from being deployed. It must check: aggregate AUC above the SLA floor, no slice regression (no LIMIT_BAL quartile degrades more than 5pp vs current production), and inference speed within the SLA latency.


### Exercise 5.3 -- Validation gate tests

Write three pytest tests for validate_new_model: (a) passes all checks, (b) fails aggregate AUC, (c) passes aggregate but fails one slice. Assert passed=True/False for each.


In [ ]:
# Exercise 5.3: Validation gate tests
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Wednesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Wednesday: The Model Validation Gate --
from sklearn.metrics import roc_auc_score
import mlflow.sklearn, pandas as pd

def validate_new_model(new_run_id, current_run_id, holdout_path):
    """Gate: new model must pass all checks to proceed to Staging."""
    new_m  = mlflow.sklearn.load_model(f'runs:/{new_run_id}/model')
    curr_m = mlflow.sklearn.load_model(f'runs:/{current_run_id}/model')
    holdout = pd.read_csv(holdout_path)
    X = holdout.drop('DEFAULT_NEXT_MONTH', axis=1)
    y = holdout['DEFAULT_NEXT_MONTH']
    new_auc  = roc_auc_score(y, new_m.predict_proba(X)[:, 1])
    curr_auc = roc_auc_score(y, curr_m.predict_proba(X)[:, 1])
    checks = {
        'auc_floor':     new_auc >= 0.80,
        'no_regression': new_auc >= curr_auc - 0.01,
    }
    # Per-slice: each LIMIT_BAL quartile must be >= 0.75
    for q_name, mask in get_limit_quartile_masks(holdout).items():
        if mask.sum() >= 20:
            s = roc_auc_score(y[mask], new_m.predict_proba(X[mask])[:, 1])
            checks[f'slice_{q_name}'] = s >= 0.75
    return all(checks.values()), checks


### Expected Output (illustrative)

`validate_new_model(...)` returns `(False, {...})` whenever any check dict value is `False` —
for example:

```
(False, {'auc_floor': True, 'no_regression': True, 'slice_q1': True,
         'slice_q2': True, 'slice_q3': False, 'slice_q4': True})
```
One failing slice is enough to fail the whole gate (`all(checks.values())`) — a model can pass
its aggregate AUC floor and still be blocked because it regressed for one credit-limit
quartile. That's the point of per-slice gating.



---

## THURSDAY -- "Automated Rollback"


Rollback must be automatic. When validation fails: archive the new model in MLflow, keep current production unchanged, alert on-call with the specific failed check.

Pause and Predict -- what minimum information must the rollback alert contain so the on-call engineer can decide whether to investigate at 3am or wait until morning? Write your answer before reading the code.


### STOP AND TRACE

Before running:

@dag(...)
def pipeline():
    t = check_triggers()
    d = fetch_training_data(t)
    m = train_model(d)
    v = validate_and_promote(m)

Draw the dependency graph. Can fetch_training_data run before check_triggers completes?
If train_model raises an exception, does validate_and_promote still run?
Why this matters: Airflow runs a task only after all upstream tasks succeed. An exception in train_model marks it FAILED and skips all downstream tasks. That is correct -- you do not want to call validate_and_promote when there is no new model.


In [ ]:
# Exercise 5.4: STOP AND TRACE -- Airflow task dependencies
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Thursday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Thursday: Automated Rollback --
from mlflow.tracking import MlflowClient

def handle_validation_result(passed, new_version, failed_checks):
    client = MlflowClient()
    if passed:
        client.transition_model_version_stage(
            name='credit-default-v2', version=new_version, stage='Staging')
        send_slack_message(
            f'Retraining PASSED: v{new_version} in Staging. Manual review for Production.')
    else:
        failed = [k for k, v in failed_checks.items() if not v]
        client.transition_model_version_stage(
            name='credit-default-v2', version=new_version, stage='Archived')
        send_pagerduty_alert(
            title='Credit model retraining FAILED',
            body=f'v{new_version} failed: {failed}. '
                 'Production model unchanged. Investigate before next Monday run.')


### Expected Output (illustrative — Slack/PagerDuty)

On a passing run:
```
Slack: "Retraining PASSED: v4 in Staging. Manual review for Production."
```
On a failing run:
```
PagerDuty: "Credit model retraining FAILED. v4 failed: ['slice_high_limit']. 
            Production model unchanged. Investigate before next Monday run."
```
Notice the failure path explicitly states *"Production model unchanged"* — the automated
system's default behaviour on failure is to do nothing to production, not to guess.



---

## FRIDAY -- "The Friday Build: End-to-End DAG Test"


Trigger the DAG manually, simulating the Week 2 drift event (PSI 0.38 > 0.25). Verify each stage in sequence. Then simulate a failed validation by setting the AUC threshold to 0.95. Verify the rollback fires: new model is Archived, current Production version unchanged.


**Friday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Friday: The Friday Build: End-to-End DAG Test --
# Manual trigger
airflow dags trigger credit_model_retraining
# Check Airflow UI: all four tasks should be green

# Verify MLflow state
from mlflow.tracking import MlflowClient
client = MlflowClient()
staging = client.get_latest_versions('credit-default-v2', stages=['Staging'])
print(f'In Staging: {[m.version for m in staging]}')

# Simulate failed validation (set threshold to 0.95)
# Retrigger -- validate_and_promote should archive the new model
prod = client.get_latest_versions('credit-default-v2', stages=['Production'])
print(f'Production unchanged: {[m.version for m in prod]}')


### Expected Output (illustrative — Airflow + MLflow)

```
$ airflow dags trigger credit_model_retraining
Created <DagRun credit_model_retraining @ 2026-06-08T02:00:00>

In Staging: [4]
Production unchanged: [3]
```
Then deliberately set the validation threshold to 0.95 and retrigger — a model that would
have passed at 0.80 should now fail the gate, and `Production unchanged` should still read
`[3]`. If it doesn't, the gate isn't actually blocking anything.


### Friday Workplace Moment

*DataFlow -- Friday DAG review.*

**Sola Fashola:** The per-slice check. Walk me through it.

**Temi Adeyemi:** checks[f'slice_{q_name}'] = s >= 0.75. One check per LIMIT_BAL quartile. Any quartile below 0.75 fails the gate.

**Sola Fashola:** If the gate fails?

**Temi Adeyemi:** passed = all(checks.values()). Any False means rollback. PagerDuty alert names the specific failed check. Production is untouched.

**Dr. Emeka Obi:** That is a usable pipeline. Merge to staging branch.



## YOUR WEEK 5 CHECKLIST

Check each item before moving on.

- [ ] I can explain the three trigger strategies and when each is appropriate.
- [ ] The Airflow DAG runs end-to-end on a manual trigger without errors.
- [ ] The validation gate checks aggregate AUC and per-slice AUC for LIMIT_BAL quartiles.
- [ ] Rollback fires automatically on validation failure with no human intervention.
- [ ] Every task logs its result to MLflow so the run is fully auditable.


## Challenge Task

> **Core path:** Implement champion-challenger evaluation using McNemar's test. Promote only if the new model is statistically significantly better (p < 0.05).

> **Advanced path:** Add a data quality gate before the retrain task: Great Expectations failures halt the pipeline before training begins.


## Common Mistakes

**Retraining without a validation gate:** A pipeline that promotes without comparing new vs current will eventually deploy a worse model.

**Validation on aggregate AUC only:** A model can improve aggregate AUC while degrading on a critical slice. Validate on the slices that matter.

**Silent rollback:** A rollback that does not alert means no one investigates the failure. The engineer must know immediately, and know which check failed.



---

**The MLOps Internship** -- Book 4 of 9, InternshipInABook™

Dataset: CreditDefault-NG | Company: DataFlow Infrastructure, Lagos/Abuja

[internshipinabook.com](https://internshipinabook.com)


📘 **Get the complete illustrated book (all 13 weeks, DOCX + EPUB):** [https://selar.com/7m27877571](https://selar.com/7m27877571)